In [1]:
# ==============================================================================
# 1. INITIAL PREPARATION AND EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================

# Import required libraries for data manipulation and machine learning
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.neighbors import BallTree

In [2]:
# Load the bike trip dataset from a CSV file
df = pd.read_csv("bike_trip_data(July 2025-June 2026).csv")

In [3]:
# Drop the 'Unnamed: 0' column which is a leftover index from a previous export
df = df.drop(columns=['Unnamed: 0'])

In [4]:
# ------------------------------------------------------------------------------
# DATA AUDIT SECTION (To be displayed in HTML tables)
# ------------------------------------------------------------------------------

data_summary = []
data_missing = []
space_str = []
data_coor_outlier = []
ins_archive = []
ins_0 = []

df_sum = pd.DataFrame(data_summary, columns=['Condition','Value'])
df_miss = pd.DataFrame(data_missing, columns=['Variables','Total_Missing'])
df_space = pd.DataFrame(space_str, columns=['Variables', 'Value with Extra Space'])
df_coor_outlier = pd.DataFrame(space_str, columns=['Variables', 'Outlier'])
df_arch = pd.DataFrame(ins_archive, columns=['variables', 'val_w_archive'])
df_0 = pd.DataFrame(ins_0, columns=['variables', 'val_w_0'])
df_dtype = pd.DataFrame(df.dtypes, columns=['type']).reset_index()
df_dtype.rename(columns={'index':'Variables'}, inplace=True)

# Check total records, duplicate data, and time input errors (start time > end time)
df_sum.loc[len(df_sum)] = ['Total Records', df.shape[0]]
df_sum.loc[len(df_sum)] = [f'Duplicate Data', df.duplicated().sum()]
df_sum.loc[len(df_sum)] = [f'Need to Swap ("started_at" & "ended_at Value")', (df['started_at'] > df['ended_at']).sum()]

# Count the number of missing values in each column
for miss in df.isna():
    df_miss.loc[len(df_miss)] = [miss, df[miss].isna().sum()]

# Check for text with extra spaces (leading, trailing, or double spaces)
for col in df.columns:
    has_extra_space = df[col].astype(str).str.contains(r'^\s+|\s+$|\s{2,}', regex=True)
    df_bermasalah = df[has_extra_space] # Menampilkan baris yang memiliki spasi ekstra
    df_space.loc[len(df_space)] = [col,len(df_bermasalah)]

# Check for outliers in latitude and longitude coordinates
for col in df.columns[8:12]:
    if col[-3:] == 'lat':
        outlier_lat = (round(df[col], 2) <= 41.60) | (round(df[col], 2) >= 42.10)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lat.sum()]
    else:
        outlier_lng = (round(df[col], 2) <= -88.00) | (round(df[col], 2) >= -87.50)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lng.sum()]

# Check for station values containing the text "Archive"
for col in df.columns[4:8]:
    archived = df[col].str.contains('Archive', case = False, na=False).sum()
    df_arch.loc[len(df_arch)] = [col,archived]

# Check for station values starting with the number "0"
for col in df.columns[4:8]:
    archived = df[col].str.startswith('0', na=False).sum()
    df_0.loc[len(df_0)] = [col,archived]

In [5]:
# Display the audit results using HTML format
html_str_1 = f"""
<h1> Dataset Check </h1>
<div style="display: grid; gap: 20px;">
    <div style="display: flex; gap: 20px;">
        <div><h3>Summary</h3>{df_sum.to_html()}</div>
        <div><h3>Coordinate Outlier</h3>{df_coor_outlier.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Missing Value</h3>{df_miss.to_html()}</div>
        <div><h3>Extra Spaces</h3>{df_space.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Value with "Archive"</h3>{df_arch.to_html()}</div>
        <div><h3>Value start with "0"</h3>{df_0.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Data Type</h3>{df_dtype.to_html()}</div>
    </div>
</div>
"""

display(HTML(html_str_1))

,Condition,Value
0,Total Records,5932350
1,Duplicate Data,35
2,"Need to Swap (""started_at"" & ""ended_at Value"")",29
,Variables,Outlier
0,start_lat,0
1,start_lng,0
2,end_lat,18
3,end_lng,8
,Variables,Total_Missing
0,ride_id,1


#### Cleaning Tasks
1. **Remove Duplicates**
    - Remove 35 duplicate records from the dataset

2. **Fix Time Recording Errors**
    - Swap `started_at` and `ended_at` values for 29 records where start time is after end time

3. **Remove or Handle Missing Values**
    - Remove records with missing end station coordinates (`end_lat` and `end_lng`) - 5,607 records (6 records already remove because its duplicated)
    - Decide on handling missing station names and IDs (1.2M+ records)

4. **Text Cleaning and Standardization**
    - Strip leading and trailing spaces from station name
    - Remove "[Archive]" tags and associated random numbers from station values
    - Convert all station names to lowercase for consistency

5. **Remove Coordinate Outliers**
    - Remove 18 records with invalid ending latitude values
    - Remove 8 records with invalid ending longitude values

6. **Rename Columns**
    - Rename `ride_id` to `trip_id`
    - Rename `rideable_type` to `bike_type`

7. **Remove Columns**
    - Remove `start_station_id` and `end_station_id` columns

    
#### Expected Outcome
A clean, standardized dataset ready for exploratory data analysis and visualization.

In [6]:
# ==============================================================================
# 2. DATA CLEANING
# ==============================================================================

# Create a copy of the data so the original dataframe is not modified
df_cleaned = df

In [7]:

# Standardize station names to lowercase and remove duplicate data
df_cleaned['start_station_name'] = df['start_station_name'].str.lower()
df_cleaned['end_station_name'] = df['end_station_name'].str.lower()
df_cleaned = df.drop_duplicates()

In [8]:
# Fix time columns: Swap values if the start time is later than the end time
condition = df_cleaned['started_at'] > df_cleaned['ended_at']
df_cleaned.loc[condition, ['started_at', 'ended_at']] = df_cleaned.loc[condition, ['ended_at', 'started_at']].values

In [9]:
# Drop rows that are missing both the end station name and its coordinates
condition = df_cleaned[['end_station_name', 'end_lat']].isna().all(axis=1)
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [10]:
# Remove the station name with the word '[archived]' in it
df_cleaned['start_station_name'] = df_cleaned['start_station_name'].str.replace('[archived]', '', regex=False)
df_cleaned['end_station_name'] = df_cleaned['end_station_name'].str.replace('[archived]', '',regex=False)

In [11]:
# ==============================================================================
# 3. STATION NAME IMPUTATION USING MACHINE LEARNING (K-NN)
# ==============================================================================

# ----------------- START STATION IMPUTATION -----------------
# Create a reference table containing unique coordinates for each start station name
station_ref = df_cleaned[
    df_cleaned['start_station_name'].notna() & 
    df_cleaned['start_station_id'].notna()
].drop_duplicates(subset=['start_lat', 'start_lng'])[
    ['start_lat', 'start_lng', 'start_station_name', 'start_station_id']
]

station_ref = station_ref.drop_duplicates(subset=['start_station_id'])
station_ref = station_ref.drop_duplicates(subset=['start_station_name'])
station_ref = station_ref.reset_index()
station_ref = station_ref.drop(columns=['index'])

In [12]:
# Prepare the Machine Learning model (BallTree) based on coordinates (Haversine distance)
mask_null = df_cleaned['start_station_name'].isna()
df_null = df_cleaned[mask_null].copy()
latlon_ref = np.radians(station_ref[['start_lat', 'start_lng']].values)
latlon_null = np.radians(df_null[['start_lat', 'start_lng']].values)
tree = BallTree(latlon_ref, metric='haversine')

In [13]:
# Find the 2 nearest stations for each point with a missing station name
distances, indices = tree.query(latlon_null, k=2)

In [14]:
# Convert distances from radians to kilometers
distances_km = distances * 6371.0
max_distances_km = 1.8
margin_km = 0.005

In [15]:
station_result = []

# Evaluate the nearest station prediction results
for i in range(len(df_null)):
    dist1, dist2 = distances_km[i][0], distances_km[i][1]
    idx1, idx2 = indices[i][0], indices[i][1]

    # If the nearest station is more than 1.8 km away, leave it blank (NaN)
    if dist1 > max_distances_km:
        station_result.append(np.nan)

    # If the distance difference between candidate 1 and 2 is very small, mark as ambiguous
    elif (dist2-dist1) <= margin_km:
        name1 = station_ref.iloc[idx1]['start_station_name']
        name2 = station_ref.iloc[idx2]['start_station_name']
        station_result.append(f'Ambiguous: {name1} / {name2}')

    # If safe, fill with the name of the first nearest station
    else:
        station_result.append(station_ref.iloc[idx1]['start_station_name'])

# Insert the prediction results back into the main dataframe
df_cleaned.loc[mask_null, 'start_station_name'] = station_result

In [16]:
# Drop rows where the start station name is still missing after prediction
condition = df_cleaned['start_station_name'].isna()
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [17]:
# Drop station ID columns since they will no be used
df_cleaned = df_cleaned.drop(columns=['start_station_id', 'end_station_id'])

In [18]:
# ----------------- END STATION IMPUTATION -----------------
# Create a reference table for end stations
station_ref = df_cleaned[
    df_cleaned['end_station_name'].notna()
].drop_duplicates(subset=['end_lat', 'end_lng'])[
    ['end_lat', 'end_lng', 'end_station_name']
]

station_ref = station_ref.drop_duplicates(subset=['end_station_name'])
station_ref = station_ref.reset_index()
station_ref = station_ref.drop(columns=['index'])

In [19]:
# Prepare the Machine Learning model for end stations
mask_null = df_cleaned['end_station_name'].isna()
df_null = df_cleaned[mask_null].copy()

latlon_ref = np.radians(station_ref[['end_lat', 'end_lng']].values)
latlon_null = np.radians(df_null[['end_lat', 'end_lng']].values)

tree = BallTree(latlon_ref, metric='haversine')

In [20]:
distances, indices = tree.query(latlon_null, k=2)

In [21]:
distances_km = distances * 6371.0

max_distances_km = 1.8
margin_km = 0.005

In [22]:
station_result = []

# Evaluate the prediction results for end stations
for i in range(len(df_null)):
    dist1, dist2 = distances_km[i][0], distances_km[i][1]
    idx1, idx2 = indices[i][0], indices[i][1]

    if dist1 > max_distances_km:
        station_result.append(np.nan)

    elif (dist2-dist1) <= margin_km:
        name1 = station_ref.iloc[idx1]['end_station_name']
        name2 = station_ref.iloc[idx2]['end_station_name']
        station_result.append(f'Ambiguous: {name1} / {name2}')

    else:
        station_result.append(station_ref.iloc[idx1]['end_station_name'])

df_cleaned.loc[mask_null, 'end_station_name'] = station_result

In [23]:
# Drop rows where the end station failed to be predicted (still missing)
condition = df_cleaned['end_station_name'].isna()
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [24]:
# ==============================================================================
# 4. ADVANCED TEXT CLEANING & DATA TYPE STANDARDIZATION
# ==============================================================================

# Clean extra spaces in station name columns (leading, trailing, and double spaces)
for col in df_cleaned.columns[4:6]:     # Access the 'start_station_name' and 'end_station_name' columns
    df_cleaned[col] = df_cleaned[col].str.strip()
    df_cleaned[col] = df_cleaned[col].str.replace(r'^\s+|\s+$|\s{2,}', ' ', regex=True)

In [25]:
# Rename columns to better describe the data content
df_cleaned.rename(columns={'ride_id': 'trip_id', 'rideable_type': 'bike_type'}, inplace=True)

In [26]:
# ==============================================================================
# 5. DATA EXPORT
# ==============================================================================

# Save the final cleaned data to a new CSV file without including the index
df_cleaned.to_csv("bike_trip_data_clean(July 2025-June 2026).csv", index=False)